In [ ]:
import pandas as pd

from matplotlib import pyplot as plt 
%matplotlib inline

import seaborn as sb

from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# path to the file to read
pcos_file_path = 'data\data without infertility _final.csv'

pcos_data = pd.read_csv(pcos_file_path) 


# JUST PLAYING AROUND TO SE HOW EVERYTHING WORSK

## Data info

In [ ]:
# Print summary statistics in next line
pcos_data.describe()

#pcos_data.columns

In [ ]:
pcos_data.info()

In [ ]:
pcos_data.columns = pcos_data.columns.str.strip()

## Cleaning data

In [ ]:
#to see empty data ; in our case it is col 42
sb.heatmap(pcos_data.isnull())

In [ ]:
#dropping col that has all null values, and then id cols cause they are not needed for the clasification
#axis=1 is for col axis=0 is for rows
pcos_data.drop(["Unnamed: 42", "Sl. No", "Patient File No."], axis=1, inplace=True)

In [ ]:
sb.heatmap(pcos_data.isnull())

Changing irregural cycles values

In [ ]:
pcos_data["Cycle(R/I)"] = pcos_data["Cycle(R/I)"].map({
    2: 0,  # Regular
    4: 1   # Irregular
})


playing again with visuals


In [ ]:
cols = pcos_data.select_dtypes(include=["int64", "float64"]).columns
cols = cols.drop("PCOS (Y/N)")  # don’t plot target vs itself

for label in cols:
    plt.figure(figsize=(6, 4))

    data_pcos = pcos_data[pcos_data["PCOS (Y/N)"] == 1][label]
    data_no_pcos = pcos_data[pcos_data["PCOS (Y/N)"] == 0][label]

    bins = 30

    plt.hist(data_pcos, bins=bins, alpha=0.6, density=True, label="PCOS", color="red")
    plt.hist(data_no_pcos, bins=bins, alpha=0.6, density=True, label="No PCOS", color="blue")

    plt.title(f"Distribution of {label}")
    plt.xlabel(label)
    plt.ylabel("Density")
    plt.legend()
    plt.show()


#### Setting features and X

In [ ]:
hormone_features = [
    "FSH(mIU/mL)",
    "LH(mIU/mL)",
    "FSH/LH",
    "AMH(ng/mL)",
    "TSH (mIU/L)",
    "PRL(ng/mL)",
    "PRG(ng/mL)"  
]


In [ ]:
patient_observable_features = [
    "Cycle(R/I)",             # Regular/Irregular periods
    "Cycle length(days)",     # length of period
    "Weight gain(Y/N)",
    "hair growth(Y/N)",
    "Skin darkening (Y/N)",
    "Hair loss(Y/N)",
    "Pimples(Y/N)",
    "Fast food (Y/N)",
    "Reg.Exercise(Y/N)"       # lifestyle activity
]

In [ ]:
doctor_observable_features = [
    # Patient-reported symptoms
    "Cycle(R/I)",
    "Cycle length(days)",
    "Weight gain(Y/N)",
    "hair growth(Y/N)",
    "Skin darkening (Y/N)",
    "Hair loss(Y/N)",
    "Pimples(Y/N)",
    "Fast food (Y/N)",
    "Reg.Exercise(Y/N)",
    
    # Doctor-observed measurements
    "Weight (Kg)",
    "Height(Cm)",
    "BMI",
    "Pulse rate(bpm)",
    "RR (breaths/min)",
    "Hb(g/dl)",
    "Hip(inch)",
    "Waist(inch)",
    "Waist:Hip Ratio",
    "BP _Systolic (mmHg)",
    "BP _Diastolic (mmHg)",
    
    # Ultrasound / gynecologist measurements
    "Follicle No. (L)",
    "Follicle No. (R)",
    "Avg. F size (L) (mm)",
    "Avg. F size (R) (mm)",
    "Endometrium (mm)"
]


In [ ]:
# all features except ids and target
# all_features = pcos_data.drop(columns=['PCOS (Y/N)'], axis=1)

all_features = list(pcos_data.drop(columns=['PCOS (Y/N)']).columns)

print(all_features)



In [ ]:
pcos_features = hormone_features

In [ ]:
# dropna drops missing values (think of na as "not available")
pcos_data = pcos_data.dropna(
    subset=pcos_features + ["PCOS (Y/N)"]
)
#drop a row only if one of features or pcos label is missing

### Separating predictors and targets

In [ ]:
# By convention, this data (features) is called X.
X = pcos_data[pcos_features]

In [ ]:
sb.heatmap(X.isnull())
plt.title("Missing values in model features")
plt.show()


In [ ]:
# want to predict if paitent has pcos or no
y = pcos_data["PCOS (Y/N)"]

In [ ]:
X.describe()

In [ ]:
X.head()

In [ ]:
#just playing to see data
plt.scatter(
    pcos_data["BMI"],
    pcos_data["PCOS (Y/N)"],
    marker='+'
)

plt.xlabel("BMI")
plt.ylabel("PCOS (Y/N)")
plt.title("BMI vs PCOS")

plt.show()

In [ ]:
y.value_counts()
# inbalanced data set 

In [ ]:

y.value_counts().plot.pie(autopct='%.2f')

In [ ]:
X.dtypes


## Random Undersampling
so that we balance the set

In [ ]:
rus = RandomUnderSampler(sampling_strategy=0.8)
X_res, y_res = rus.fit_resample(X, y)

ax = y_res.value_counts().plot.pie(autopct='%.2f')
_ = ax.set_title("Under-sampling")

In [ ]:
y_res.value_counts()

## Normalize the data 

In [ ]:
#create a scaler object so that we can normalize the data
scaler = StandardScaler()

# fit the scaler to the data and then transform the data
X_scaled = scaler.fit_transform(X_res)

X_scaled

## Spliting data into train and test

In [ ]:
from sklearn.model_selection import train_test_split

#the function train_test_split returns 4 diff values
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_res, test_size=0.25, random_state=67)
# if you ever want to repeat the same split, you have to repeat random_state num

# always change this value when you change it in the function so that you can log it later
test_size=0.25

## Train the model

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression()

#actually train the model
logistic_model.fit(X_train, y_train)

In [ ]:
#predict the target on test data
predictions = logistic_model.predict(X_test)

print(predictions)

### Evaluation 

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy: .4f}")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

## Logs how each split and feature list perfomed


In [ ]:
import os
import pandas as pd

log_path = r'logs\experiment_log.csv'

# Try reading the log; if fails, create empty DataFrame
try:
    experiment_log = pd.read_csv(log_path)
except (pd.errors.EmptyDataError, FileNotFoundError):
    experiment_log = pd.DataFrame(columns=["Features", "Test Size", "Accuracy", "Notes"])

# Your experiment info
notes = "Logistic Regression model - on all features"
new_entry = pd.DataFrame([{
    "Features": ', '.join(pcos_features),
    "Test Size": test_size,
    "Accuracy": round(accuracy, 4),
    "Notes": notes
}])

# Append new entry
experiment_log = pd.concat([experiment_log, new_entry], ignore_index=True)

# Ensure folder exists
os.makedirs(os.path.dirname(log_path), exist_ok=True)

# Save log
experiment_log.to_csv(log_path, index=False)

print("Experiment logged successfully!")
